In [1]:
import pandas as pd
import random
import numpy as np
from tqdm import tqdm
import ipdb
import re

import matplotlib.pyplot as plt
# import mplcursors
import seaborn as sns
%matplotlib inline
sns.set(style='darkgrid', context='notebook', rc={'figure.figsize':(14,10)}, font_scale=2)

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('chained_assignment',None)

# Set random seeds for reproducibility on a specific machine
random.seed(1)
np.random.seed(1)
np.random.RandomState(1)

RandomState(MT19937) at 0x120220540

In [13]:
df = pd.read_csv('date_counts_mini.tsv', sep='\t')

In [14]:
df.head()

,month,day,iso,date_label,count_total,count_full_month_day,count_full_month_ordinal,count_abbr_month_day,count_abbr_month_ordinal
0,1,1,01-01,January 1,15333618,9707113,1815530,3542481,268494
1,1,2,01-02,January 2,6232804,3730066,434825,1977707,90206
2,1,3,01-03,January 3,6873284,4140023,434398,2206547,92316
3,1,4,01-04,January 4,6779335,4048710,409043,2230994,90588
4,1,5,01-05,January 5,6783237,4106928,426472,2158161,91676


In [15]:
df.sort_values(by=['count_total'], ascending=False).head(10)

,month,day,iso,date_label,count_total,count_full_month_day,count_full_month_ordinal,count_abbr_month_day,count_abbr_month_ordinal
254,9,11,09-11,September 11,20447222,11993019,2911449,5324398,218356
0,1,1,01-01,January 1,15333618,9707113,1815530,3542481,268494
182,7,1,07-01,July 1,11123409,8149841,1154783,1763523,55262
91,4,1,04-01,April 1,9831697,6726070,1187939,1854788,62900
365,12,31,12-31,December 31,9683282,6170066,883239,2502790,127187
274,10,1,10-01,October 1,9525194,5658804,889223,2837582,139585
121,5,1,05-01,May 1,9357306,8231874,1125432,0,0
181,6,30,06-30,June 30,9090214,6595684,781441,1654473,58616
60,3,1,03-01,March 1,9042253,5952122,864490,2159708,65933
152,6,1,06-01,June 1,8871965,6172304,856808,1789012,53841


In [16]:
df.sort_values(by=['count_total']).head(10)

,month,day,iso,date_label,count_total,count_full_month_day,count_full_month_ordinal,count_abbr_month_day,count_abbr_month_ordinal
59,2,29,02-29,February 29,2220002,1302064,202741,672396,42801
358,12,24,12-24,December 24,4622130,2922303,322014,1311694,66119
360,12,26,12-26,December 26,4794688,3033140,319166,1380465,61917
359,12,25,12-25,December 25,4809135,2836266,746717,1137310,88842
361,12,27,12-27,December 27,5136665,3209039,263228,1603480,60918
363,12,29,12-29,December 29,5452725,3443125,269114,1674455,66031
357,12,23,12-23,December 23,5486603,3445163,331646,1637285,72509
362,12,28,12-28,December 28,5489011,3417921,283549,1721828,65713
364,12,30,12-30,December 30,5726314,3626502,309841,1720470,69501
328,11,24,11-24,November 24,5827841,3605038,310479,1834264,78060


## Months that are mentioned most often

In [17]:
df.groupby('month', group_keys=False)[['count_total']].mean().sort_values(by='count_total', ascending=False)

,count_total
month,
1,7.460365e+06
9,7.423471e+06
4,7.117586e+06
6,7.016250e+06
3,6.925068e+06
7,6.850671e+06
11,6.772372e+06
2,6.753855e+06
5,6.735819e+06


### Days mentioned most often

In [20]:
df.groupby('day', group_keys=False)[['count_total']].median().sort_values(by='count_total', ascending=False).head(10)

,count_total
day,
1,8957109.0
31,7599532.0
15,7252249.0
20,7096990.0
4,6956898.5
6,6946911.0
7,6910413.5
22,6905830.5
12,6901374.0


### Dates mentioned least often

In [23]:
df.groupby('day', group_keys=False)[['count_total']].median().sort_values(by='count_total').head(10)

,count_total
day,
16,6562806.0
13,6563771.0
29,6571228.0
28,6580860.5
24,6605591.5
27,6610751.0
26,6632683.5
17,6655892.0
23,6677941.5


### Most common dates in different formats

In [24]:
df.sort_values(by=['count_full_month_day'], ascending=False).head(5)

,month,day,iso,date_label,count_total,count_full_month_day,count_full_month_ordinal,count_abbr_month_day,count_abbr_month_ordinal
254,9,11,09-11,September 11,20447222,11993019,2911449,5324398,218356
0,1,1,01-01,January 1,15333618,9707113,1815530,3542481,268494
121,5,1,05-01,May 1,9357306,8231874,1125432,0,0
182,7,1,07-01,July 1,11123409,8149841,1154783,1763523,55262
91,4,1,04-01,April 1,9831697,6726070,1187939,1854788,62900


In [25]:
df.sort_values(by=['count_full_month_ordinal'], ascending=False).head(5)

,month,day,iso,date_label,count_total,count_full_month_day,count_full_month_ordinal,count_abbr_month_day,count_abbr_month_ordinal
254,9,11,09-11,September 11,20447222,11993019,2911449,5324398,218356
0,1,1,01-01,January 1,15333618,9707113,1815530,3542481,268494
185,7,4,07-04,July 4,8258453,5143468,1657657,1411232,46096
91,4,1,04-01,April 1,9831697,6726070,1187939,1854788,62900
182,7,1,07-01,July 1,11123409,8149841,1154783,1763523,55262


In [26]:
df[df.month!=5].sort_values(by=['count_abbr_month_day'], ascending=False).head(5)

,month,day,iso,date_label,count_total,count_full_month_day,count_full_month_ordinal,count_abbr_month_day,count_abbr_month_ordinal
254,9,11,09-11,September 11,20447222,11993019,2911449,5324398,218356
0,1,1,01-01,January 1,15333618,9707113,1815530,3542481,268494
274,10,1,10-01,October 1,9525194,5658804,889223,2837582,139585
31,2,1,02-01,February 1,8312992,4883877,561559,2720156,147400
5,1,6,01-06,January 6,8366844,4953086,727308,2567162,119288


In [27]:
df[df.month!=5].sort_values(by=['count_abbr_month_ordinal'], ascending=False).head(5)

,month,day,iso,date_label,count_total,count_full_month_day,count_full_month_ordinal,count_abbr_month_day,count_abbr_month_ordinal
0,1,1,01-01,January 1,15333618,9707113,1815530,3542481,268494
254,9,11,09-11,September 11,20447222,11993019,2911449,5324398,218356
31,2,1,02-01,February 1,8312992,4883877,561559,2720156,147400
244,9,1,09-01,September 1,8746876,5370731,770248,2463513,142384
274,10,1,10-01,October 1,9525194,5658804,889223,2837582,139585


### Diversity of date forms

In [29]:
form_cols = [
    "count_full_month_day",
    "count_full_month_ordinal",
    "count_abbr_month_day",
    "count_abbr_month_ordinal",
]

def form_evenness(row, cols=form_cols):
    if row.month==5:
        cols = ["count_full_month_day", "count_full_month_ordinal"]
    counts = row[cols].to_numpy(dtype=float)
    total = row['count_total']

    p = counts / total
    # nonzero mask avoids 0*log(0) warnings
    nz = p > 0
    H = -np.sum(p[nz] * np.log(p[nz]))
    return H

df["form_evenness"] = df.apply(form_evenness, axis=1)

In [33]:
df.sort_values(by=['form_evenness'], ascending=False).head(5)

,month,day,iso,date_label,count_total,count_full_month_day,count_full_month_ordinal,count_abbr_month_day,count_abbr_month_ordinal,form_evenness
359,12,25,12-25,December 25,4809135,2836266,746717,1137310,88842,1.015337
254,9,11,09-11,September 11,20447222,11993019,2911449,5324398,218356,0.989327
59,2,29,02-29,February 29,2220002,1302064,202741,672396,42801,0.969405
274,10,1,10-01,October 1,9525194,5658804,889223,2837582,139585,0.953377
0,1,1,01-01,January 1,15333618,9707113,1815530,3542481,268494,0.951393


In [35]:
df[df.month!=5].sort_values(by=['form_evenness']).head(5)

,month,day,iso,date_label,count_total,count_full_month_day,count_full_month_ordinal,count_abbr_month_day,count_abbr_month_ordinal,form_evenness
182,7,1,07-01,July 1,11123409,8149841,1154783,1763523,55262,0.781404
181,6,30,06-30,June 30,9090214,6595684,781441,1654473,58616,0.786307
163,6,12,06-12,June 12,7182450,4999589,456941,1674167,51753,0.802442
173,6,22,06-22,June 22,7029915,4878382,429374,1662521,59638,0.805738
167,6,16,06-16,June 16,6426920,4444650,429775,1500230,52265,0.814674
